### Initial Inference

In [ ]:
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
import torch

#Inference function
def predict_document_type_debug(conversations, max_tokens=128, temperature=0.8, top_p=0.95):
    """
    Return both cleaned and raw model outputs for inspection.
    """
    if not conversations or not isinstance(conversations, list):
        print("[DEBUG] No conversation content found.")
        return "no_input", None

    user_prompt = conversations[0].get("content", "")
    if not user_prompt.strip():
        print("[DEBUG] User prompt is empty.")
        return "no_input", None

    messages = [{"role": "user", "content": user_prompt}]

    # Apply the chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
    )

    # Decode raw output with special tokens
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]

    # Extract response
    if "<|start_header_id|>assistant<|end_header_id|>" in decoded_output:
        response = decoded_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    else:
        response = decoded_output

    # Clean common special tokens
    cleaned = (
        response.replace("<|eot_id|>", "")
                .replace("<|end_of_turn|>", "")
                .strip()
    )

    return cleaned, decoded_output


In [ ]:
cleaned, raw  = predict_document_type_debug(val_dataset[120]["conversations"])
print(cleaned)

In [ ]:
val_dataset[120]["conversations"]

In [ ]:
sample = {
    "conversations": [
        {
            "role": "user",
            "content": '[USER]: You are an expert in identifying document types using OCR text. Your role is to look at the OCR texts inputs provided by the user and classify them into different document types. \
You will return the response in Output format defined below.\n\n\
The possible document types are: [\'mill_certificate\', \'invoice\', \'bill_of_lading\', \'packing_list\'].\n\n\
Example:\n\
Input: \'{"ocr": ["page 1 text", "page 2 text"], "num_pages": 2}\'\n\
Output: \'{"bill_of_lading": [[0], [1]]}\'\n\n\
Input: {"ocr": ["Packing List\nExporter: Global Trade Ltd\nConsignee: Star Imports Inc\nContents:\n- 100 units Widget A\n- 50 units Widget B\nTotal weight: 550kg"], "num_pages": 1}\n\n\
Output format: \'{"document_type": [[page_numbers]]}\'\n\n\
Your response should only contain the string in the provided output format and no other text.\n\n\
[ASSISTANT]:'
        }
    ]
}

### answer should be packing_list: [[0]]

cleaned, raw  = predict_document_type_debug(sample["conversations"])
print(cleaned)

In [ ]:
sample = {
    "conversations": [
        {
            "role": "user",
            "content": '''[USER]: You are an expert in identifying document types using OCR text. Your role is to look at the OCR texts inputs provided by the user and classify them into different document types. 
You will return the response in Output format defined below.

The possible document types are: ['mill_certificate', 'invoice', 'bill_of_lading', 'packing_list'].

Example:
Input: '{"ocr": ["page 1 text", "page 2 text"], "num_pages": 2}'
Output: '{"bill_of_lading": [[0], [1]]}'

Input: {"ocr": [
"Invoice\nINVOICE #4567\nDate: 2024-11-12\nSeller: Alpha Electronics Ltd.\nBuyer: Tech World Co.\nItems:\n- 10x SSD 1TB @ $100\n- 5x Monitor 24\" @ $150\nTotal Amount Due: $1,750\nPayment Terms: Net 30 Days",
"Mill Certificate\nCertificate No: 9982\nManufacturer: SteelCorp Industries\nProduct: Cold Rolled Steel Sheets\nSpecification: ASTM A1008\nHeat No: 558930\nMechanical Properties:\n- Yield Strength: 280 MPa\n- Tensile Strength: 420 MPa\nCertified by: QA Engineer - John Smith"
], "num_pages": 2}

Output format: '{"document_type": [[page_numbers]]}'

Your response should only contain the string in the provided output format and no other text.

[ASSISTANT]:'''
        }
    ]
}

###output should be {"invoice": [[0]], "mill_certificate": [[1]]}

cleaned, raw  = predict_document_type_debug(sample["conversations"])
print(cleaned)

### Evaluating

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import numpy as np
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from evaluator import DocumentEvaluator, evaluate_strict_document_groups
import json
import time
from tqdm import tqdm

def is_valid_json(text):
    try:
        json.loads(text)
        return True
    except Exception:
        return False

def run_combined_evaluation(dataset, predict_fn, log_top_k_slowest=3, iou_threshold=1.0):
    evaluator = DocumentEvaluator()
    raw_preds = []
    raw_trues = []
    invalid_logs = []
    inference_times = []

    for i in tqdm(range(len(dataset)), desc="Evaluating"):
        sample = dataset[i]

        # Extract true response
        true_text = ""
        for turn in sample.get('conversations', []):
            if turn.get('role') == 'assistant':
                true_text = turn.get('content', "")
                break

        # Run inference
        start_time = time.time()
        pred_text = predict_fn(sample)
        elapsed = time.time() - start_time
        inference_times.append((i, elapsed))

        if not is_valid_json(pred_text):
            invalid_logs.append({
                "index": i,
                "input": sample.get("conversations", [])[0]["content"],
                "expected": true_text,
                "prediction": pred_text
            })
            continue

        raw_preds.append(pred_text)
        raw_trues.append(true_text)
        evaluator.add_sample(pred_text, true_text)

    # Save invalid predictions
    with open("invalid_predictions_log.txt", "w", encoding="utf-8") as f:
        for item in invalid_logs:
            f.write(f"--- Sample {item['index']} ---\n")
            f.write("Input:\n" + item["input"] + "\n")
            f.write("Expected:\n" + item["expected"] + "\n")
            f.write("Prediction:\n" + item["prediction"] + "\n\n")

    # Run evaluations
    print("\n===== LEVEL 1: Document Classification =====")
    level1_metrics = evaluator.evaluate()

    print("===== LEVEL 2 & 3: Group-Level Evaluation =====")
    level23_metrics = evaluate_strict_document_groups(raw_preds, raw_trues, iou_threshold=iou_threshold)

    total_time = sum(t for _, t in inference_times)
    valid_samples = len(raw_preds)
    all_samples = len(dataset)

    print("\n===== Inference Timing =====")
    print(f"Total time: {total_time:.2f} sec")
    print(f"Avg time per valid sample: {total_time / max(valid_samples, 1):.2f} sec")
    print(f"Avg time per all samples: {total_time / all_samples:.2f} sec")

    print(f"\n===== Top {log_top_k_slowest} Slowest Samples =====")
    inference_times.sort(key=lambda x: x[1], reverse=True)
    for i, t in inference_times[:log_top_k_slowest]:
        print(f"Sample {i}: {t:.2f} sec")

    return {
        "document_level": level1_metrics,
        "segmentation_level": level23_metrics,
        "timing": {
            "total_seconds": total_time,
            "avg_per_valid_sample": total_time / max(valid_samples, 1),
            "avg_per_all_sample": total_time / all_samples,
            "slowest_samples": inference_times[:log_top_k_slowest]
        },
        "invalid_samples": invalid_logs,
    }

def wrapped_predict_fn(sample):
    return predict_document_type_debug(sample["conversations"])[0]


In [ ]:
results = run_combined_evaluation(val_dataset, wrapped_predict_fn)
results

### Evaluation-Observation (Testing)

In [ ]:
## somthings off doc-level prediction maybe? --check later

# Sample prediction and ground truth
pred_str = '{"invoice": [[0]], "packing_list": [[1]]}'
true_str = '{"invoice": [[0]], "bill_of_lading": [[1]]}'

from evaluator import DocumentEvaluator, evaluate_strict_document_groups

# Create evaluator instance
evaluator = DocumentEvaluator()

# Add single sample
evaluator.add_sample(pred_str, true_str)

# Run document-level evaluation
doc_metrics = evaluator.evaluate()

# Run group-level evaluation
group_metrics = evaluate_strict_document_groups([pred_str], [true_str])

# Print both
print("\n📊 Document-Level Metrics:\n", doc_metrics)
print("\n📦 Group-Level Metrics:\n", group_metrics)